# Assignment 4 — Mini Myanmar ASR

**Student:** Kyaw Zaya Naing  
**Course:** AIE-F-B2 · [assignment-4](https://github.com/ye-kyaw-thu/AIE-F-B2/tree/main/assignment/assignment-4)

## ဤ notebook တွင် လုပ်မည့်အဆင့်များ

| အဆင့် | အကြောင်းအရာ |
|------|-------------|
| **A** | Data Prep — speaker-held-out train / dev / test |
| **B** | Kaldi baseline — Monophone + Triphone HMM-GMM |
| **C** | Advanced — Kaldi DNN-HMM + PyTorch BiLSTM-CTC |
| **D** | Evaluation — Character WER + Syllable Error Rate (SER) |

> ရှည်လျားသော train implementation သည် `scripts/` ထဲတွင်ရှိသည်။  
> Notebook က အဆင့်လိုက် ခေါ်သုံးပြီး ရလဒ်ကို ပြသည် (`01_data_prep` / `02_kaldi_train` ပုံစံ)။

---
# Part A — Data Preparation

ရည်ရွယ်ချက်: `raw-corpus/` ထဲက အသံဖိုင်များကို Kaldi သုံးနိုင်သော  
`kaldi_data/{train,dev,test}` အဖြစ် ပြင်ဆင်ရန်။

### Assignment စည်းမျဉ်း (speaker-held-out)

- Train / Dev / Test ကို **utterance မဟုတ်ဘဲ speaker (လူ)** အလိုက် ခွဲသည်
- **Test** တွင် ယောက်ျား ၁ ဦး + မိန်းမ ၁ ဦး ကို **လုံးဝ held-out** ထားသည် (train ထဲ မပါ)
- ထိုသို့ ခွဲမှသာ model က မကြားဖူးသော အသံကို မှန်မှန်ကန်ကန် အသိအမှတ်ပြုနိုင်မှုကို တိုင်းနိုင်သည်

### A.1 · လမ်းကြောင်းနှင့် environment

In [ ]:
from pathlib import Path
import json
import subprocess

ROOT = Path("/home/administrator/assignmen-kzyn/KyawZayaNaing-assignment4").resolve()
RAW = ROOT / "raw-corpus"
OUT = ROOT / "kaldi_data"
EXP = ROOT / "kaldi_recipe" / "exp"
LOGS = ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

CONDA = Path.home() / "miniconda3" / "etc" / "profile.d" / "conda.sh"
ENV = "kaldi-asr"

SCRIPTS = {
    "prep": ROOT / "scripts" / "prepare_kaldi_data.py",
    "gmm":  ROOT / "scripts" / "kaldi_mini_train.py",
    "dnn":  ROOT / "scripts" / "kaldi_dnn_train.py",
    "ctc":  ROOT / "scripts" / "pytorch_ctc_train.py",
}

print("ROOT   :", ROOT)
print("RAW    :", RAW, "exists=", RAW.exists())
print("scripts:")
for k, p in SCRIPTS.items():
    print(f"  {k:4s} → {p.name}  ({'OK' if p.exists() else 'MISSING'})")


### A.2 · Speaker split သတ်မှတ်ချက်

| Split | Speakers | Gender | မှတ်ချက် |
|-------|----------|--------|---------|
| **Train** | BoyClone, PunKhattgar, HanZawHtet | M / F / M | own corpus only |
| **Dev** | YinMon | F | hyperparameter / early-stop |
| **Test** | KyawZayaNaing, NawSayRiHtoo | **M + F** | held-out အဆုံးသတ် တိုင်းတာရန် |

> SPLIT အဓိက သတ်မှတ်ချက်သည် `scripts/prepare_kaldi_data.py` ထဲတွင်ရှိသည်။  
> အောက် cell တွင် လက်ရှိ config ကို ဖတ်ပြသည်။

In [ ]:
# prepare script ထဲက SPLIT ကို notebook မှာလည်း ရှင်းအောင် ပြထားသည်
SPLIT = {
    # Own corpus only — do not import speakers from other projects
    "train": ["BoyClone", "PunKhattgar", "HanZawHtet"],
    "dev":   ["YinMon"],
    "test":  ["KyawZayaNaing", "NawSayRiHtoo"],  # 1 male + 1 female held-out
}
GENDER = {
    "BoyClone": "male",
    "HanZawHtet": "male",
    "PunKhattgar": "female",
    "YinMon": "female",
    "KyawZayaNaing": "male",
    "NawSayRiHtoo": "female",
}
print("Speaker-held-out SPLIT:")
print(json.dumps(SPLIT, ensure_ascii=False, indent=2))
print("\nTest genders:", {s: GENDER[s] for s in SPLIT["test"]})
assert set(GENDER[s] for s in SPLIT["test"]) >= {"male", "female"}, "test must be 1M+1F"


### A.3 · Script ခေါ်ရန် helper

`kaldi-asr` conda env အတွင်း script များ run ရန်။ Log များကို `logs/` ထဲ သိမ်းသည်။

In [ ]:
def run_py(script: Path, *args: str, log: str = "run.log", env: str | None = ENV) -> None:
    """Run a Python script (optionally inside conda env) and tee output to logs/."""
    log_path = LOGS / log
    arg_s = " ".join(args)
    cmd = "\n".join([
        f'source "{CONDA}"',
        f"conda activate {env}" if env else "true",
        f'cd "{ROOT}"',
        f'python "{script}" {arg_s} 2>&1 | tee "{log_path}"',
    ])
    print("Running:", script.name, arg_s)
    print("Log    :", log_path)
    proc = subprocess.run(["bash", "-lc", cmd], text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"failed rc={proc.returncode}; see {log_path}")
    print("OK\n")


def show_scores(model: str, split: str = "test") -> dict | None:
    p = EXP / model / f"decode_{split}" / "scores.json"
    if not p.exists():
        print(f"{model}/{split}: (no scores yet)")
        return None
    s = json.loads(p.read_text(encoding="utf-8"))
    print(
        f"{model}/{split}:  Char WER={s['char_wer']*100:.1f}%  "
        f"SER={s['syllable_er']*100:.1f}%  n={s.get('n_utt')}"
    )
    return s


### A.4 · Kaldi data ဖန်တီးခြင်း

`prepare_kaldi_data.py` က လုပ်သည်မှာ —

1. `raw-corpus/` အောက် nested Rec folder များရှာသည် (`*_Rec1` … `Rec5`, YinMon `001`–`005`)
2. `recordings.tsv` + `.wav` မှ utterance စုသည် (16 kHz mono)
3. Speaker SPLIT အတိုင်း `train` / `dev` / `test` ခွဲသည်
4. Kaldi files ရေးသည် — `wav.scp`, `text`, `utt2spk`, `spk2utt`

In [ ]:
run_py(SCRIPTS["prep"], log="01_prepare.log", env=None)

print("=" * 60)
print((OUT / "split_plan.md").read_text(encoding="utf-8"))
cfg = json.loads((OUT / "split_config.json").read_text(encoding="utf-8"))
print("counts :", cfg["counts"])
print("gender :", cfg["gender"])


### A.5 · Data integrity စစ်ဆေးခြင်း

- utterance ID ထပ်နေမှု မရှိရ (Kaldi `compute-cmvn-stats` error ဖြစ်နိုင်)
- `wav.scp` / `text` / `utt2spk` ID များ တူညီရမည်
- Test speakers သည် train ထဲ **မပါ** ရ

In [ ]:
def check_split(name: str) -> dict:
    d = OUT / name
    wav_ids = [ln.split()[0] for ln in (d / "wav.scp").read_text().splitlines() if ln.strip()]
    text_ids = [ln.split()[0] for ln in (d / "text").read_text().splitlines() if ln.strip()]
    u2s = dict(ln.split() for ln in (d / "utt2spk").read_text().splitlines() if ln.strip())

    assert set(wav_ids) == set(text_ids) == set(u2s), f"{name}: ID mismatch"
    assert len(wav_ids) == len(set(wav_ids)), f"{name}: duplicate utt-ids"
    speakers = sorted(set(u2s.values()))
    print(f"{name:5s}  utterances={len(wav_ids):4d}  speakers={speakers}")
    return {"n": len(wav_ids), "speakers": speakers}


stats = {s: check_split(s) for s in ("train", "dev", "test")}

# held-out အတည်ပြု
leak = set(stats["train"]["speakers"]) & set(stats["test"]["speakers"])
assert not leak, f"TEST speakers leaked into TRAIN: {leak}"
print("\nHeld-out OK — test speakers not in train.")


---
# Part B — Kaldi HMM-GMM Baseline

ရည်ရွယ်ချက်: monophone နှင့် triphone **HMM-GMM** baseline model နှစ်ခု လေ့ကျင့်ပြီး  
test set ပေါ် Char WER / SER တိုင်းရန်။

| Stage | လုပ်ဆောင်ချက် |
|-------|-------------|
| 0–2 | data copy, character lexicon, MFCC + CMVN + deltas |
| 3 | **Monophone** HMM-GMM |
| 4 | **Triphone** HMM-GMM (context-dependent) |
| 5 | Decode test/dev + score |

### B.1 · Features & lexicon (stage 0–2)

In [ ]:
run_py(
    SCRIPTS["gmm"],
    "--stage", "0", "--stop-stage", "2",
    log="02_kaldi_feat.log",
)
print("feats ready:", (ROOT / "kaldi_recipe" / "data" / "train" / "feats.scp").exists())


### B.2 · Monophone HMM-GMM (stage 3)

အခြေခံဆုံး acoustic model — phone တစ်ခုချင်းစီကို context မကြည့်ဘဲ model လုပ်သည်။  
Baseline အဖြစ် သုံးသည်။

In [ ]:
MONO_ITERS = 20  # smoke test လုပ်ချင်ရင် 8 ထားနိုင်
run_py(
    SCRIPTS["gmm"],
    "--stage", "3", "--stop-stage", "3",
    "--mono-iters", str(MONO_ITERS),
    log="03_kaldi_mono.log",
)
print("mono/final.mdl:", (EXP / "mono" / "final.mdl").exists())


### B.3 · Triphone HMM-GMM (stage 4)

ဘယ်/ညာ phone context ကို ထည့်သွင်းသော model။  
ပုံမှန်အားဖြင့် monophone ထက် ပိုကောင်းသင့်သော်လည်း data နည်းရင် မတက်နိုင်။

In [ ]:
TRI_ITERS = 15
try:
    run_py(
        SCRIPTS["gmm"],
        "--stage", "4", "--stop-stage", "4",
        "--tri-iters", str(TRI_ITERS),
        log="04_kaldi_tri.log",
    )
except Exception as e:
    print("Triphone failed (optional):", e)
print("tri1/final.mdl:", (EXP / "tri1" / "final.mdl").exists())


### B.4 · Decode & score (stage 5)

Test (နှင့် dev) ကို decode လုပ်ပြီး —

- **Char WER** — မြန်မာစာလုံး (character) အလိုက် edit distance
- **SER** — syllable အလိုက် error rate

In [ ]:
run_py(
    SCRIPTS["gmm"],
    "--stage", "5", "--stop-stage", "5",
    "--decode-split", "test",
    log="05_kaldi_decode.log",
)

print("--- GMM baselines (test) ---")
show_scores("mono", "test")
show_scores("tri1", "test")
print("--- GMM (dev) ---")
show_scores("mono", "dev")
show_scores("tri1", "dev")


---
# Part C — Advanced Models

Assignment လိုအပ်ချက်: GMM baseline ထက် **ပိုကောင်းသော** model ≥ ၁ ခု။

| Model | ဖော်ပြချက် |
|-------|-----------|
| **Kaldi DNN-HMM** | Triphone alignment ပေါ် feed-forward DNN (nnet2) |
| **PyTorch BiLSTM-CTC** | MFCC → BiLSTM → CTC loss → greedy decode |

### C.1 · Kaldi DNN-HMM

Triphone (`tri1`) alignment ကို အသုံးပြုပြီး pdf ခန့်မှန်းသော DNN။  
Graph သည် triphone HCLG ကို ဆက်သုံးသည်။

In [ ]:
if (EXP / "tri1" / "final.mdl").exists():
    run_py(
        SCRIPTS["dnn"],
        "--stage", "0", "--epochs", "15",
        log="06_kaldi_dnn.log",
    )
    show_scores("dnn", "test")
else:
    print("Skip DNN — tri1/final.mdl မရှိသေးပါ။ Part B.3 ကို အရင် run ပါ။")


### C.2 · PyTorch BiLSTM-CTC

End-to-end character model —

1. Kaldi MFCC + CMVN + deltas ကို input အဖြစ် ထုတ်သည်  
2. Bidirectional LSTM + CTC loss ဖြင့် လေ့ကျင့်သည်  
3. Greedy decode → Char WER / SER

In [ ]:
# torch + kaldiio လိုအပ် — base/current env
run_py(SCRIPTS["ctc"], log="07_ctc.log", env=None)
show_scores("ctc", "test")
print("checkpoint:", (EXP / "ctc" / "best.pt").exists())


---
# Part D — Results & Discussion

Test set (KyawZayaNaing + NawSayRiHtoo) ပေါ်ရှိ model များ နှိုင်းယှဉ်ချက်။

### D.1 · Score table

In [ ]:
rows = []
for name, key in [
    ("GMM Monophone", "mono"),
    ("GMM Triphone", "tri1"),
    ("Kaldi DNN-HMM", "dnn"),
    ("PyTorch BiLSTM-CTC", "ctc"),
]:
    p = EXP / key / "decode_test" / "scores.json"
    if not p.exists():
        rows.append((name, None, None, None))
        continue
    s = json.loads(p.read_text(encoding="utf-8"))
    rows.append((name, s.get("char_wer"), s.get("syllable_er"), s.get("n_utt")))

print(f"{'Model':<22} {'Char WER':>10} {'SER':>10} {'#utt':>6}")
print("-" * 52)
for name, wer, ser, n in rows:
    w = f"{wer*100:9.1f}%" if wer is not None else "      —"
    s = f"{ser*100:9.1f}%" if ser is not None else "      —"
    u = f"{n:6d}" if n is not None else "     —"
    print(f"{name:<22} {w} {s} {u}")

# save summary
summary = [
    {"model": n, "char_wer": w, "syllable_er": s, "n_utt": u}
    for n, w, s, u in rows
]
(EXP / "summary_test_scores.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("\nWrote", EXP / "summary_test_scores.json")


### D.2 · Sample hypotheses (အကောင်းဆုံး model)

In [ ]:
def load_text(path: Path) -> dict[str, str]:
    out = {}
    if not path.exists():
        return out
    for ln in path.read_text(encoding="utf-8").splitlines():
        parts = ln.split(maxsplit=1)
        if len(parts) == 2:
            out[parts[0]] = parts[1]
    return out

# CTC > DNN > mono > tri1 အစဉ်အတိုင်း ရှိသည့် decode ကို ပြ
pref = next(
    (m for m in ("ctc", "dnn", "mono", "tri1")
     if (EXP / m / "decode_test" / "trans.txt").exists()),
    None,
)
print("Showing hypotheses from:", pref)
if pref:
    hyp = load_text(EXP / pref / "decode_test" / "trans.txt")
    ref = load_text(ROOT / "kaldi_recipe" / "data" / "test" / "text_orig")
    for utt in sorted(set(hyp) & set(ref))[:6]:
        print("UTT", utt)
        print("REF", ref[utt])
        print("HYP", hyp[utt])
        print("---")


### D.3 · Discussion (ဆရာ့အတွက် အကျဉ်း)

1. **Split** — speaker-held-out; test = 1 male (`KyawZayaNaing`) + 1 female (`NawSayRiHtoo`)။  
   Train speakers သည် test ထဲ လုံးဝ မပါ။

2. **Baselines** — Monophone / Triphone HMM-GMM။ Data နည်းပြီး character-level lexicon ဖြစ်သောကြောင့်  
   GMM WER မြင့်တတ်သည်။

3. **Advanced** — DNN-HMM နှင့် BiLSTM-CTC။ CTC သည် ပုံမှန်အားဖြင့် ဤ mini corpus တွင် GMM ထက် ပိုကောင်းတတ်သည်။

4. **Metrics** — Char WER (စာလုံး) နှင့် SER (syllable) နှစ်မျိုးလုံး တိုင်းထားသည်။

5. **Scripts**
   - `scripts/prepare_kaldi_data.py` — data prep  
   - `scripts/kaldi_mini_train.py` — GMM train / decode  
   - `scripts/kaldi_dnn_train.py` — DNN-HMM  
   - `scripts/pytorch_ctc_train.py` — BiLSTM-CTC  

6. Speaker / SPLIT ပြောင်းလျှင် Part A မှ ပြန် run ပါ။